# Notebook 2 · Feature Selection for Linear Models

Learn which input columns help predict y (and which are traps), using the same 11-step recipe with many features.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rathachai/DA-LAB/blob/gh-pages/learnings/linear/notebooks/nb-02-feature-selection.ipynb)

- Lecture: [en-02-feature-selection.md](https://github.com/Rathachai/DA-LAB/blob/gh-pages/learnings/linear/en-02-feature-selection.md)
- Lab: [CORR101](https://rathachai.github.io/DA-LAB/learnings/linear/corr101.html) · [LM201](https://rathachai.github.io/DA-LAB/learnings/linear/lm201.html)

The same 11-step recipe every time:

1. Load libraries
2. Get data
3. Display data
4. Visualize correlation
5. Process data
6. Select X and y
7. Split into train and test
8. Create and train the model
9. Predict
10. Evaluate
11. Visualize y and y_pred

Powered by: Sonnet  |  AI Whisperer: rathachai.github.io

## Step 1 · Load libraries
Same toolbox as before: pandas, numpy, scikit-learn, and matplotlib for plots.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (mean_absolute_error, mean_squared_error,
                             mean_absolute_percentage_error, r2_score)
pd.options.display.float_format = '{:.3f}'.format
print("Libraries ready")

## Step 2 · Get data
Now the table has many candidate inputs: `id` (row label), `x1` to `x7`, and the target `y`. Which columns really help?

In [ ]:
url = "https://rathachai.github.io/DA-LAB/learnings/linear/data/lm201_data.csv"
df = pd.read_csv(url)
df

## Step 3 · Display data
Check size and typical values of every column.

In [ ]:
print("rows, columns:", df.shape)
df.describe()

## Step 4 · Visualize correlation
Pearson **r** (-1 to +1) scores only *straight-line* links with `y`. Rank the columns by |r|, but always look at the plots too: a column can matter and still have r near 0.

In [ ]:
r_y = df.drop(columns="y").corrwith(df["y"]).rename("r with y")
display(r_y.to_frame())
r_y.plot.bar(color=["tab:green" if v > 0 else "tab:red" for v in r_y])
plt.axhline(0, color="black", lw=0.8); plt.title("Pearson r of each column with y")
plt.ylabel("r"); plt.show()

Scatter plots tell the rest of the story. `x3` is a **U-shape** (r is about 0 but it clearly matters), `x4` is **pure noise**, `x5` is a **curve**, and `id` is just a label.

In [ ]:
cols = ["id", "x1", "x2", "x3", "x4", "x5", "x6", "x7"]
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for ax, col in zip(axes.ravel(), cols):
    ax.scatter(df[col], df["y"], s=8)
    ax.set_title(f"{col}  (r = {df[col].corr(df['y']):.2f})"); ax.set_xlabel(col); ax.set_ylabel("y")
plt.tight_layout(); plt.show()

## Step 5 · Process data
`x5` curves upward (exponential growth). Taking its logarithm straightens it, like reading a decibel scale instead of raw power. We add a new column `ln_x5`.

In [ ]:
df["ln_x5"] = np.log(df["x5"])
df

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
for a, col in zip(ax, ["x5", "ln_x5"]):
    a.scatter(df[col], df["y"], s=10)
    a.set_title(f"{col} vs y  (r = {df[col].corr(df['y']):.2f})"); a.set_xlabel(col); a.set_ylabel("y")
plt.tight_layout(); plt.show()

## Step 6 · Select X and y
Keep the useful columns (`x1`, `x2`, `x6`, `x7`, `ln_x5`). Leave out `id` (a label), `x4` (noise), `x3` (U-shape, no straight-line signal) and the raw `x5` (we use its log).

In [ ]:
features = ["x1", "x2", "ln_x5", "x6", "x7"]
X = df[features]
y = df["y"]
print("X shape:", X.shape, "| y shape:", y.shape)
X

## Step 7 · Split into train and test
No split in this chapter (it is the next topic). We evaluate on the same data we trained on, so the scores are optimistic.

## Step 8 · Create and train the model
With several inputs the line becomes `y = c + m1*x1 + m2*x2 + ...`. Each coefficient says how much y changes when that one input grows by 1 and the others stay fixed.

In [ ]:
model = LinearRegression()
model.fit(X, y)
print("intercept c =", round(model.intercept_, 3))
pd.DataFrame({"coefficient": model.coef_}, index=features)

## Step 9 · Predict
Predict `y` for every row (`y_pred`), shown next to the actual value.

In [ ]:
y_pred = model.predict(X)
pd.DataFrame({"y": y, "y_pred": y_pred, "residual": y - y_pred}).head()

## Step 10 · Evaluate
**MAE** = average miss, **RMSE** = penalises big misses, **MAPE** = miss in % of y, **R²** = share of y's variation explained (1 = perfect). Scored on the training data.

In [ ]:
def evaluate(y, y_pred):
    return pd.DataFrame({"value": [
        mean_absolute_error(y, y_pred),
        np.sqrt(mean_squared_error(y, y_pred)),
        100 * mean_absolute_percentage_error(y, y_pred),
        r2_score(y, y_pred)]},
        index=["MAE", "RMSE", "MAPE (%)", "R2"])
evaluate(y, y_pred)

## Step 11 · Visualize y and y_pred
There is no single x-axis with five inputs, so we plot actual vs predicted. Points on the dashed diagonal `y = y_pred` are perfect predictions.

In [ ]:
plt.scatter(y, y_pred)
lim = [y.min(), y.max()]
plt.plot(lim, lim, "k--", label="y = y_pred")
plt.xlabel("actual y"); plt.ylabel("predicted y")
plt.title("Actual vs predicted"); plt.legend(); plt.show()

## Compare feature sets
Same recipe, different columns. We fit each set and keep R², adjusted R² and MAE in one table.

In [ ]:
sets = {"x1 only": ["x1"],
        "x1 + x7": ["x1", "x7"],
        "good set": features,
        "good + x4 (noise)": features + ["x4"],
        "good + x3 (U-shape)": features + ["x3"],
        "all incl. id": ["id", "x1", "x2", "x3", "x4", "x5", "x6", "x7"]}
rows, n = {}, len(df)
for name, cols in sets.items():
    p = LinearRegression().fit(df[cols], y).predict(df[cols])
    r2 = r2_score(y, p)
    rows[name] = {"R2": r2, "adj R2": 1 - (1 - r2) * (n - 1) / (n - len(cols) - 1),
                  "MAE": mean_absolute_error(y, p)}
results = pd.DataFrame(rows).T
results

In [ ]:
results[["R2", "adj R2"]].plot.bar(figsize=(9, 4), ylim=(0, 1))
plt.title("R² and adjusted R² by feature set"); plt.ylabel("score")
plt.xticks(rotation=30, ha="right"); plt.tight_layout(); plt.show()

**R² vs adjusted R².** R² never goes down when you add a column, even a useless one, so it can reward clutter.
Adjusted R² subtracts a penalty for each extra column, so noise like `x4` does not look like progress.
Compare feature sets with adjusted R² (and MAE), and prefer the smaller set when scores tie.

## Summary

- Follow the same 11 steps every time; only the data and the columns change.
- r finds straight-line links only: `x3` (U-shape) hides, `x4` is noise, `id` is a label; plot before you trust r.
- A log transform (`ln_x5`) straightens a curved feature and helps the model.
- R² always rises with more columns; use adjusted R² to compare feature sets.

Powered by: Sonnet  |  AI Whisperer: rathachai.github.io